# Telco Customer Churn Prediction

An end-to-end machine learning project that analyses customer data and predicts the likelihood of customer churn using **Logistic Regression, Random Forest, and XGBoost**.

## Project Overview

This notebook follows the complete data science workflow: it defines the customer churn problem, explores and understands the dataset, cleans missing and inconsistent data, and prepares the features for modelling. It then performs a stratified train/validation/test split and builds a preprocessing pipeline for numerical and categorical features, including imputation, scaling, and encoding. Next, it trains and tunes multiple classification models, evaluates and compares their performance using relevant classification metrics, and selects a decision threshold using validation data. Finally, it uses feature-importance analysis and SHAP explainability to interpret the model's predictions and identify factors associated with customer churn. The selected model and its metadata are exported for use in the Streamlit application.

**Note:** The test set is kept separate for the final model evaluation and is not used for model selection or threshold tuning.

## 1. Imports and configuration

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, classification_report,
    confusion_matrix, RocCurveDisplay, PrecisionRecallDisplay
)
from sklearn.inspection import permutation_importance

from xgboost import XGBClassifier

RANDOM_STATE = 42
DATASET_NAME = "Telco_Customer_Churn"
DATA_PATH = Path("Telco_Customer_Churn.csv")
TARGET = "Churn"

## 2. Load the dataset

In [ ]:
# Resolve relative to the current notebook working directory.
# If running from another folder, update DATA_PATH to the CSV's full path.
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Cannot find '{DATA_PATH.resolve()}'. "
        "Place Telco_Customer_Churn.csv in the same folder as this notebook."
    )

data = pd.read_csv(DATA_PATH)
print(f"{DATASET_NAME} loaded successfully.")
print("Shape:", data.shape)
display(data.head())

In [ ]:
print("Dataset information:")
data.info()

print("\\nMissing values:")
display(data.isna().sum().sort_values(ascending=False).to_frame("Missing values"))

print("\\nDuplicate rows:", data.duplicated().sum())
print("\\nChurn distribution:")
display(data["Churn"].value_counts().to_frame("Customers"))

## 3. Data cleaning and initial exploration

In [ ]:
data = data.copy()

# TotalCharges is stored as text in the source file; blank strings become NaN.
data["TotalCharges"] = pd.to_numeric(data["TotalCharges"], errors="coerce")

# Remove exact duplicate rows, if any.
data = data.drop_duplicates().reset_index(drop=True)

print("Rows after duplicate removal:", len(data))
print("Blank TotalCharges values:", data["TotalCharges"].isna().sum())
print("Churn rate:", round((data["Churn"] == "Yes").mean() * 100, 2), "%")

In [ ]:
# Numeric summary
display(data[["tenure", "MonthlyCharges", "TotalCharges"]].describe().T)

# Target distribution
churn_counts = data["Churn"].value_counts().reindex(["No", "Yes"], fill_value=0)
plt.figure(figsize=(6, 4))
plt.bar(churn_counts.index, churn_counts.values)
plt.title("Customer Churn Distribution")
plt.xlabel("Churn")
plt.ylabel("Number of customers")
plt.tight_layout()
plt.show()

## 4. Exploratory data analysis and key dataset insights

In [ ]:
# Create analysis-only helper columns. They are not automatically used as model features.
eda = data.copy()
eda["TenureGroup"] = pd.cut(
    eda["tenure"],
    bins=[-1, 12, 24, 48, 60, 1000],
    labels=["0-12 months", "13-24 months", "25-48 months", "49-60 months", "60+ months"]
)

churn_rate = (eda["Churn"] == "Yes").mean()
print(f"Overall churn rate: {churn_rate:.1%}")
print(f"Customers: {len(eda):,}")
print(f"Average tenure: {eda['tenure'].mean():.1f} months")
print(f"Average monthly charge: ${eda['MonthlyCharges'].mean():.2f}")
print(f"Median monthly charge: ${eda['MonthlyCharges'].median():.2f}")

In [ ]:
def churn_rate_table(column):
    return (
        eda.groupby(column, observed=False)["Churn"]
        .apply(lambda s: (s == "Yes").mean())
        .mul(100)
        .sort_values(ascending=False)
        .rename("Churn rate (%)")
        .to_frame()
    )

for column in ["Contract", "TenureGroup", "PaymentMethod", "InternetService", "TechSupport", "OnlineSecurity"]:
    print(f"Churn rate by {column}:")
    display(churn_rate_table(column).round(1))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

contract_plot = churn_rate_table("Contract")["Churn rate (%)"]
axes[0, 0].bar(contract_plot.index.astype(str), contract_plot.values)
axes[0, 0].set_title("Churn Rate by Contract")
axes[0, 0].set_ylabel("Churn rate (%)")
axes[0, 0].tick_params(axis="x", rotation=15)

tenure_plot = churn_rate_table("TenureGroup")["Churn rate (%)"]
axes[0, 1].bar(tenure_plot.index.astype(str), tenure_plot.values)
axes[0, 1].set_title("Churn Rate by Tenure")
axes[0, 1].set_ylabel("Churn rate (%)")
axes[0, 1].tick_params(axis="x", rotation=15)

payment_plot = churn_rate_table("PaymentMethod")["Churn rate (%)"]
axes[1, 0].bar(payment_plot.index.astype(str), payment_plot.values)
axes[1, 0].set_title("Churn Rate by Payment Method")
axes[1, 0].set_ylabel("Churn rate (%)")
axes[1, 0].tick_params(axis="x", rotation=20)

no_churn_charges = eda.loc[eda["Churn"] == "No", "MonthlyCharges"].dropna()
churn_charges = eda.loc[eda["Churn"] == "Yes", "MonthlyCharges"].dropna()
axes[1, 1].boxplot([no_churn_charges, churn_charges], tick_labels=["No churn", "Churn"])
axes[1, 1].set_title("Monthly Charges by Churn Status")
axes[1, 1].set_ylabel("Monthly charges")

plt.tight_layout()
plt.show()

### Generate evidence-based recommendations

In [ ]:
# The recommendations below are generated from the actual segment churn rates.
segment_columns = ["Contract", "TenureGroup", "PaymentMethod", "InternetService", "TechSupport", "OnlineSecurity"]
segment_rates = {
    col: churn_rate_table(col)["Churn rate (%)"]
    for col in segment_columns
}

recommendations = []

contract_rates = segment_rates["Contract"]
if len(contract_rates):
    highest_contract = contract_rates.idxmax()
    recommendations.append(
        f"Contract strategy: '{highest_contract}' has the highest observed churn "
        f"({contract_rates.max():.1f}%). Test retention offers and clearer renewal/value messaging "
        "for customers in this contract group."
    )

tenure_rates = segment_rates["TenureGroup"]
if len(tenure_rates):
    highest_tenure = tenure_rates.idxmax()
    recommendations.append(
        f"Early-life retention: the '{highest_tenure}' tenure segment has the highest churn "
        f"({tenure_rates.max():.1f}%). Consider onboarding check-ins, setup assistance, and "
        "proactive support during this period."
    )

payment_rates = segment_rates["PaymentMethod"]
if len(payment_rates):
    highest_payment = payment_rates.idxmax()
    recommendations.append(
        f"Payment experience: '{highest_payment}' customers show the highest churn "
        f"({payment_rates.max():.1f}%). Investigate payment friction and offer convenient "
        "autopay/payment alternatives without assuming payment method causes churn."
    )

for feature in ["TechSupport", "OnlineSecurity"]:
    rates = segment_rates[feature]
    if "No" in rates.index and rates["No"] > rates.max() - 0.001:
        recommendations.append(
            f"Service support: customers recorded as '{feature} = No' have a churn rate "
            f"of {rates['No']:.1f}%. Test whether support/security education or relevant service "
            "bundles improve retention."
        )

print("DATA-DRIVEN RETENTION RECOMMENDATIONS")
for i, item in enumerate(recommendations, 1):
    print(f"{i}. {item}")

print("\\nInterpretation note: segment differences are associations, not proof of causation.")

## 5. Prepare features and target

In [ ]:
# Identifier is excluded because it does not provide a generalizable customer behaviour signal.
drop_columns = [TARGET]
if "customerID" in data.columns:
    drop_columns.append("customerID")

X = data.drop(columns=drop_columns)
y = data[TARGET].map({"No": 0, "Yes": 1})

if y.isna().any():
    raise ValueError("Unexpected target values. Expected Churn values 'Yes' and 'No'.")

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = [c for c in X.columns if c not in numeric_features]

print("Feature count:", X.shape[1])
print("Numeric:", numeric_features)
print("Categorical:", categorical_features)
print("Class counts:")
display(y.value_counts().rename(index={0: "No churn", 1: "Churn"}).to_frame("Customers"))

## 6. Train, validation, and test split

In [ ]:
# 60% train, 20% validation, 20% test. Stratification preserves class proportions.
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25,
    stratify=y_train_val, random_state=RANDOM_STATE
)

for name, target_part in [("Train", y_train), ("Validation", y_val), ("Test", y_test)]:
    print(f"{name}: {len(target_part):,} rows | churn rate {target_part.mean():.2%}")

## 7. Preprocessing pipelines

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

# Class imbalance weight is calculated from the training partition only.
negative_count = int((y_train == 0).sum())
positive_count = int((y_train == 1).sum())
scale_pos_weight = negative_count / max(positive_count, 1)
print("XGBoost scale_pos_weight baseline:", round(scale_pos_weight, 3))

## 8. Define models and hyperparameter search spaces

In [ ]:
model_spaces = {
    "Logistic Regression": (
        Pipeline([
            ("preprocessor", preprocessor),
            ("model", LogisticRegression(
                class_weight="balanced", max_iter=2000,
                solver="liblinear", random_state=RANDOM_STATE
            ))
        ]),
        {
            "model__C": [0.01, 0.1, 0.5, 1, 5, 10],
            "model__penalty": ["l1", "l2"]
        },
        8
    ),
    "Random Forest": (
        Pipeline([
            ("preprocessor", preprocessor),
            ("model", RandomForestClassifier(
                class_weight="balanced_subsample",
                random_state=RANDOM_STATE, n_jobs=-1
            ))
        ]),
        {
            "model__n_estimators": [200, 300, 450],
            "model__max_depth": [None, 8, 12, 20],
            "model__min_samples_split": [2, 5, 10],
            "model__min_samples_leaf": [1, 2, 4],
            "model__max_features": ["sqrt", 0.5]
        },
        10
    ),
    "XGBoost": (
        Pipeline([
            ("preprocessor", preprocessor),
            ("model", XGBClassifier(
                objective="binary:logistic",
                eval_metric="logloss",
                scale_pos_weight=scale_pos_weight,
                random_state=RANDOM_STATE,
                n_jobs=-1,
                tree_method="hist"
            ))
        ]),
        {
            "model__n_estimators": [200, 300, 450, 600],
            "model__learning_rate": [0.02, 0.03, 0.05, 0.1],
            "model__max_depth": [2, 3, 4, 5],
            "model__min_child_weight": [1, 3, 5],
            "model__subsample": [0.7, 0.8, 0.9, 1.0],
            "model__colsample_bytree": [0.7, 0.8, 0.9, 1.0],
            "model__gamma": [0, 0.1, 0.3],
            "model__reg_alpha": [0, 0.1, 1],
            "model__reg_lambda": [1, 2, 5]
        },
        12
    )
}

## 9. Hyperparameter tuning

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
trained_models = {}
tuning_summary = []

for model_name, (pipeline, params, iterations) in model_spaces.items():
    print(f"\nTraining and tuning: {model_name}")
    search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=params,
        n_iter=iterations,
        scoring="roc_auc",
        cv=cv,
        n_jobs=-1,
        random_state=RANDOM_STATE,
        refit=True,
        verbose=1
    )
    search.fit(X_train, y_train)
    trained_models[model_name] = search.best_estimator_
    tuning_summary.append({
        "Model": model_name,
        "Best CV ROC-AUC": search.best_score_,
        "Best Parameters": search.best_params_
    })

tuning_df = pd.DataFrame(tuning_summary).sort_values("Best CV ROC-AUC", ascending=False)
display(tuning_df[["Model", "Best CV ROC-AUC"]].round(4))

## 10. Select decision thresholds using validation data

In [ ]:
def threshold_search(model, X_validation, y_validation):
    probabilities = model.predict_proba(X_validation)[:, 1]
    threshold_values = np.arange(0.10, 0.91, 0.01)
    rows = []
    for threshold in threshold_values:
        predicted = (probabilities >= threshold).astype(int)
        rows.append({
            "threshold": threshold,
            "precision": precision_score(y_validation, predicted, zero_division=0),
            "recall": recall_score(y_validation, predicted, zero_division=0),
            "f1": f1_score(y_validation, predicted, zero_division=0)
        })
    result = pd.DataFrame(rows)
    best_threshold = float(result.loc[result["f1"].idxmax(), "threshold"])
    return best_threshold, result

thresholds_by_model = {}
threshold_curves = {}

for name, model in trained_models.items():
    best_threshold, curve = threshold_search(model, X_val, y_val)
    thresholds_by_model[name] = best_threshold
    threshold_curves[name] = curve
    print(f"{name}: validation-selected threshold = {best_threshold:.2f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
for ax, (name, curve) in zip(axes, threshold_curves.items()):
    ax.plot(curve["threshold"], curve["precision"], label="Precision")
    ax.plot(curve["threshold"], curve["recall"], label="Recall")
    ax.plot(curve["threshold"], curve["f1"], label="F1")
    ax.axvline(thresholds_by_model[name], linestyle="--", alpha=0.7)
    ax.set_title(name)
    ax.set_xlabel("Decision threshold")
    ax.grid(alpha=0.25)
axes[0].set_ylabel("Validation score")
axes[-1].legend()
plt.tight_layout()
plt.show()

## 11. Final model evaluation on the untouched test set

In [ ]:
test_probabilities = {}
test_predictions = {}
evaluation_rows = []

for name, model in trained_models.items():
    probability = model.predict_proba(X_test)[:, 1]
    threshold = thresholds_by_model[name]
    predicted = (probability >= threshold).astype(int)

    test_probabilities[name] = probability
    test_predictions[name] = predicted

    evaluation_rows.append({
        "Model": name,
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_test, predicted),
        "Precision": precision_score(y_test, predicted, zero_division=0),
        "Recall": recall_score(y_test, predicted, zero_division=0),
        "F1": f1_score(y_test, predicted, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, probability),
        "PR-AUC": average_precision_score(y_test, probability)
    })

performance_df = pd.DataFrame(evaluation_rows).sort_values("ROC-AUC", ascending=False)
display(performance_df.round(4))

In [ ]:
for name in trained_models:
    print(f"\\n{name}")
    print(classification_report(
        y_test, test_predictions[name],
        target_names=["No Churn", "Churn"],
        zero_division=0
    ))

## 12. ROC and Precision-Recall curve comparison

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for name, probability in test_probabilities.items():
    RocCurveDisplay.from_predictions(y_test, probability, name=name, ax=ax)
ax.set_title("ROC Curve Comparison")
ax.grid(alpha=0.25)
plt.show()

fig, ax = plt.subplots(figsize=(8, 6))
for name, probability in test_probabilities.items():
    PrecisionRecallDisplay.from_predictions(y_test, probability, name=name, ax=ax)
ax.set_title("Precision-Recall Curve Comparison")
ax.grid(alpha=0.25)
plt.show()

## 13. Confusion matrix and selected model

In [ ]:
# For this demonstration artifact, select the model with the highest validation CV ROC-AUC.
# This avoids using test-set performance to choose the deployment model.
selected_model_name = tuning_df.iloc[0]["Model"]
selected_model = trained_models[selected_model_name]
selected_threshold = thresholds_by_model[selected_model_name]
selected_predictions = test_predictions[selected_model_name]

print("Selected model based on training CV ROC-AUC:", selected_model_name)
print("Threshold selected on validation data:", selected_threshold)
print("Test ROC-AUC:", round(roc_auc_score(y_test, test_probabilities[selected_model_name]), 4))

cm = confusion_matrix(y_test, selected_predictions)
plt.figure(figsize=(5, 4))
plt.imshow(cm, interpolation="nearest")
plt.title(f"Confusion Matrix — {selected_model_name}")
plt.colorbar()
tick_marks = np.arange(2)
plt.xticks(tick_marks, ["Predicted No Churn", "Predicted Churn"], rotation=15)
plt.yticks(tick_marks, ["Actual No Churn", "Actual Churn"])
for row in range(cm.shape[0]):
    for col in range(cm.shape[1]):
        plt.text(col, row, str(cm[row, col]), ha="center", va="center")
plt.tight_layout()
plt.show()

## 14. Permutation feature importance

In [ ]:
# Permutation importance uses held-out test features and ROC-AUC.
# Interpret as predictive association, not causation.
perm = permutation_importance(
    selected_model, X_test, y_test,
    scoring="roc_auc",
    n_repeats=8,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

importance_df = pd.DataFrame({
    "Feature": X_test.columns,
    "Importance mean": perm.importances_mean,
    "Importance std": perm.importances_std
}).sort_values("Importance mean", ascending=False)

display(importance_df.head(15))

plot_data = importance_df.head(15).sort_values("Importance mean")
plt.figure(figsize=(9, 6))
plt.barh(plot_data["Feature"], plot_data["Importance mean"], xerr=plot_data["Importance std"])
plt.xlabel("Decrease in ROC-AUC after permutation")
plt.title("Permutation Feature Importance")
plt.tight_layout()
plt.show()

## 15. Optional SHAP interpretation

In [ ]:
# SHAP is optional. If installed, this section explains a tree-based selected model.
# Run: pip install shap
try:
    import shap

    if selected_model_name in ["Random Forest", "XGBoost"]:
        prep = selected_model.named_steps["preprocessor"]
        tree_estimator = selected_model.named_steps["model"]
        sample_raw = X_test.sample(min(250, len(X_test)), random_state=RANDOM_STATE)
        sample_transformed = prep.transform(sample_raw)
        transformed_feature_names = prep.get_feature_names_out()

        explainer = shap.TreeExplainer(tree_estimator)
        shap_values = explainer.shap_values(sample_transformed)

        if isinstance(shap_values, list):
            values_for_plot = shap_values[1]
        elif getattr(shap_values, "ndim", 0) == 3:
            values_for_plot = shap_values[:, :, 1]
        else:
            values_for_plot = shap_values

        shap.summary_plot(
            values_for_plot, sample_transformed,
            feature_names=transformed_feature_names
        )
        shap.summary_plot(
            values_for_plot, sample_transformed,
            feature_names=transformed_feature_names,
            plot_type="bar"
        )
    else:
        print("Selected model is Logistic Regression; use permutation importance above or a linear SHAP explainer.")
except ImportError:
    print("SHAP is not installed. Install with `pip install shap` to run this optional section.")
except Exception as error:
    print("SHAP could not run in this environment:", repr(error))

## 16. Export the selected model for Streamlit

In [ ]:
artifact = {
    "model": selected_model,
    "model_name": selected_model_name,
    "threshold": selected_threshold,
    "feature_columns": X.columns.tolist(),
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "target": TARGET
}

joblib.dump(artifact, "best_churn_model.joblib")
performance_df.to_csv("model_comparison.csv", index=False)
importance_df.to_csv("feature_importance.csv", index=False)

print("Saved best_churn_model.joblib")
print("Saved model_comparison.csv")
print("Saved feature_importance.csv")

## 17. Conclusions and business recommendations

Use the EDA tables and charts above to identify the customer segments with the highest observed churn. Recommendations should be based on those measured patterns, not assumptions.

Examples of actions to test:
- Improve onboarding and proactive support for the tenure segment with the highest churn.
- Test contract renewal incentives for the contract type with the highest churn.
- Investigate billing and payment friction for the payment method with the highest churn.
- Evaluate whether customers without support/security add-ons need clearer service education or support options.

**Caution:** Segment-level associations and model feature importance do not prove that a feature causes churn. Validate retention interventions with controlled experiments where possible.